**Name :** Sarvajit Patil

**ROll No.:** DS82

**PRN :** 2425010120

**GenAI Experiment 9**

In [ ]:
%pip install -q -U langchain langchain-community langchain-google-genai duckduckgo-search

  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.62.0
    Uninstalling google-auth-2.62.0:
      Successfully uninstalled google-auth-2.62.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [google-auth]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
google-genai 2.33.0 requires google-auth[requests]<3.0.0,>=2.56.0, but you have google-auth 2.49.0 which is incompatible.


In [ ]:
#Task 1 — Configure Gemini API key securely
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass(
        "Enter your Gemini API key: "
    )

print("Gemini API key configured.")

Enter your Gemini API key: ··········
Gemini API key configured.


In [ ]:
#Step 4 — Create the web-search function

%pip install -U ddgs
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

def search_web(topic):
    if not topic or not topic.strip():
        raise ValueError("Please enter a topic.")

    results = search_tool.invoke(topic.strip())
    return str(results)

In [ ]:
results = search_web("Recent developments in artificial intelligence")
print(results)

Artificial Intelligence News. Everything on AI including futuristic robots with artificial intelligence, computer models of human intelligence and more. Discover the latest AI Innovations transforming industries and enhancing technology in the United States. Stay informed with cutting-edge developments today. This AI snapshot provides a general update on recent trends and significant developments in generative AI observed by the ACCC since the March 2025 Digital Platform Services Inquiry final report. Artificial intelligence continues to evolve at an accelerating pace, transitioning from narrow, data-driven tools to systems capable of reasoning, and autonomous action. Get the latest artificial intelligence news, breakthroughs, and analysis. AI Weekly covers model releases, startup funding, tools, and trends daily.


In [ ]:
#Step 5 — Create the Gemini summarization function

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a research assistant. Use the provided search results "
        "to answer accurately. Do not invent facts. "
        "Return exactly three concise bullet points. "
        "Do not include an introduction or conclusion."
    ),
    (
        "human",
        "Topic: {topic}\n\n"
        "Search results:\n{search_results}\n\n"
        "Write exactly three informative bullet points."
    )
])

summary_chain = summary_prompt | llm

def summarize_results(topic, search_results):
    response = summary_chain.invoke({
        "topic": topic,
        "search_results": search_results
    })

    return response.content.strip()

In [ ]:
#Step 6 — Combine searching and summarization into one agent

def run_research_agent(topic):
    print(f"[1/2] Searching the web for: {topic}")

    search_results = search_web(topic)

    print("[1/2] Search completed.")
    print("[2/2] Summarizing search results...")

    brief = summarize_results(topic, search_results)

    print("[2/2] Summarization completed.")
    print("\nRESEARCH BRIEF")
    print(brief)

    return brief

In [ ]:
#Step 7 — Test the agent with two topics
topic_1 = "Recent developments in artificial intelligence"

brief_1 = run_research_agent(topic_1)

[1/2] Searching the web for: Recent developments in artificial intelligence
[1/2] Search completed.
[2/2] Summarizing search results...
[2/2] Summarization completed.

RESEARCH BRIEF
*   Artificial intelligence is evolving rapidly, transitioning from narrow, data-driven tools to systems capable of reasoning and autonomous action.
*   AI innovations are transforming industries and enhancing technology, particularly in the United States.
*   Recent developments include significant trends and advancements in generative AI.


In [ ]:
topic_2 = "Cybersecurity best practices for students"

brief_2 = run_research_agent(topic_2)

[1/2] Searching the web for: Cybersecurity best practices for students
[1/2] Search completed.
[2/2] Summarizing search results...
[2/2] Summarization completed.

RESEARCH BRIEF
*   Students should utilize strong passwords, passkeys, and multi-factor authentication (MFA) to protect their accounts and devices.
*   Awareness of cyber threats like phishing, malware, and scams is essential for students to adopt safer online practices.
*   Practical cybersecurity measures are important for campus Wi-Fi, online collaboration, and safeguarding academic data and personal privacy.


In [ ]:
#Step 8 — Experiment with temperature
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

In [ ]:
#Observation
'''
The agent combines web search and summarization into one sequential workflow.
DuckDuckGo retrieves search results, and Gemini generates a concise research brief.
The prompt requests exactly three bullet points and excludes unnecessary introductory text.
Changing temperature can affect the consistency and variation of the generated wording.
The output quality depends on the relevance of search results and the prompt used.
'''